# Hatch Requirements.txt - Requirements File Plugin

Hatch Requirements.txt is a plugin for the Hatch build system that allows reading dependencies from requirements.txt files during the build process.

## Example 1: Plugin registration with hatchling

In [ ]:
import importlib.metadata

import hatch_requirements_txt
from hatch_requirements_txt import RequirementsMetadataHook, hatch_register_metadata_hook
from hatchling.plugin.manager import PluginManager

print("hatch-requirements-txt", importlib.metadata.version("hatch-requirements-txt"))

# The plugin registers a metadata hook named "requirements_txt"
assert RequirementsMetadataHook.PLUGIN_NAME == "requirements_txt"
assert hatch_register_metadata_hook() is RequirementsMetadataHook

# hatchling discovers the hook through its plugin manager (entry point "hatch")
manager = PluginManager()
assert manager.metadata_hook.get("requirements_txt") is RequirementsMetadataHook
print("hatchling resolves 'requirements_txt' to", RequirementsMetadataHook.__name__)

## Example 2: Parsing requirement lines with parse_requirements

In [ ]:
from hatch_requirements_txt import parse_requirements

# Comment lines are collected, pip options ("-r", "--hash") are skipped,
# and trailing comments are stripped from requirement lines.
requirements, comments = parse_requirements(["requests>=2  # c", "# comment", "-r other.txt"])
assert [r.name for r in requirements] == ["requests"]
assert str(requirements[0]) == "requests>=2"
assert comments == ["# comment"]

lines = [
    "Foo_Bar[Extra]>=1.0; python_version >= '3.8'",
    "click>=8.0.0 --hash=sha256:abc",
    "",
    "uvicorn[standard]>=0.18,!=0.19.0",
]
requirements, comments = parse_requirements(lines)
for req in requirements:
    print(f"{req.name:10} specifier={str(req.specifier):18} extras={sorted(req.extras)} marker={req.marker}")

# Names are canonicalized, extras/markers kept, pip options cut off, blank lines ignored
assert [r.name for r in requirements] == ["foo-bar", "click", "uvicorn"]
assert requirements[0].extras == {"Extra"}
assert str(requirements[0].marker) == 'python_version >= "3.8"'
assert str(requirements[1]) == "click>=8.0.0"
assert requirements[2].specifier.contains("0.20") and not requirements[2].specifier.contains("0.19.0")
assert comments == []

## Example 3: Loading files with load_requirements_files

In [ ]:
import os
import tempfile

from hatch_requirements_txt import load_requirements_files
from packaging.specifiers import SpecifierSet

with tempfile.TemporaryDirectory() as temp_dir:
    base = os.path.join(temp_dir, "requirements.txt")
    with open(base, "w", encoding="utf-8") as f:
        f.write("# Web\n")
        f.write("requests>=2.25.0  # HTTP library\n")
        f.write("\n")
        f.write("click>=8.0.0 \\\n")          # a line continued with a backslash
        f.write("    --hash=sha256:0000\n")
    extra = os.path.join(temp_dir, "requirements-db.txt")
    with open(extra, "w", encoding="utf-8") as f:
        f.write("SQLAlchemy>=1.4,<3\n")
        f.write("psycopg2-binary>=2.9; sys_platform != 'win32'\n")

    requirements, comments = load_requirements_files([base, extra])
    print([str(r) for r in requirements])
    print(comments)

    # Requirements from both files, in file order; the continuation line is unfolded
    assert [r.name for r in requirements] == ["requests", "click", "sqlalchemy", "psycopg2-binary"]
    assert requirements[1].specifier == SpecifierSet(">=8.0.0")
    assert requirements[2].specifier == SpecifierSet(">=1.4,<3")
    assert str(requirements[3].marker) == 'sys_platform != "win32"'
    assert comments == ["# Web"]

## Example 4: The metadata hook fills dependencies and optional-dependencies

In [ ]:
import os
import tempfile

from hatch_requirements_txt import RequirementsMetadataHook

with tempfile.TemporaryDirectory() as root:
    requirements_file = os.path.join(root, "requirements.txt")
    with open(requirements_file, "w", encoding="utf-8") as f:
        f.write("click>=8.0.0\n")
        f.write("requests>=2.25.0\n")
    test_file = os.path.join(root, "requirements-test.txt")
    with open(test_file, "w", encoding="utf-8") as f:
        f.write("pytest>=7\n")

    # Same table as [tool.hatch.metadata.hooks.requirements_txt] in pyproject.toml
    config = {"files": [requirements_file], "optional-dependencies": {"test": [test_file]}}
    metadata = {"name": "demo-package", "dynamic": ["dependencies", "optional-dependencies"]}

    hook = RequirementsMetadataHook(root, config)
    hook.update(metadata)
    print("dependencies:", metadata["dependencies"])
    print("optional-dependencies:", metadata["optional-dependencies"])

    assert metadata["dependencies"] == ["click>=8.0.0", "requests>=2.25.0"]
    assert metadata["optional-dependencies"] == {"test": ["pytest>=7"]}

## Example 5: Errors for invalid configuration and input

In [ ]:
import os
import tempfile

from hatch_requirements_txt import (
    RequirementsMetadataHook,
    load_requirements_files,
    parse_requirements,
)
from packaging.requirements import InvalidRequirement

with tempfile.TemporaryDirectory() as root:
    requirements_file = os.path.join(root, "requirements.txt")
    with open(requirements_file, "w", encoding="utf-8") as f:
        f.write("click>=8.0.0\n")

    def expect(exc_type, func, *args):
        try:
            func(*args)
        except exc_type as e:
            print(f"{exc_type.__name__}: {e}")
        else:
            raise AssertionError(f"expected {exc_type.__name__}")

    # 'files' is configured, but dependencies are not declared dynamic
    hook = RequirementsMetadataHook(root, {"files": [requirements_file]})
    expect(ValueError, hook.update, {"name": "demo", "dependencies": []})

    # dependencies are dynamic but also listed statically
    expect(ValueError, hook.update, {"name": "demo", "dynamic": ["dependencies"], "dependencies": ["x"]})

    # 'filename' (deprecated) and 'files' together
    both = RequirementsMetadataHook(root, {"filename": requirements_file, "files": [requirements_file]})
    expect(ValueError, both.update, {"name": "demo", "dynamic": ["dependencies"]})

    # missing file, wrong argument type, malformed requirement line
    expect(FileNotFoundError, load_requirements_files, [os.path.join(root, "missing.txt")])
    expect(TypeError, load_requirements_files, requirements_file)
    expect(InvalidRequirement, parse_requirements, ["package=="])

    # a file holding only comments and blank lines yields no requirements
    comments_only = os.path.join(root, "comments.txt")
    with open(comments_only, "w", encoding="utf-8") as f:
        f.write("# one\n\n  # two\n\n")
    requirements, comments = load_requirements_files([comments_only])
    assert requirements == [] and comments == ["# one", "  # two"]